In [1]:
import tensorflow as tf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

2025-02-16 18:42:31.170439: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-02-16 18:42:31.180795: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1739711551.192645   44797 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1739711551.196171   44797 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-02-16 18:42:31.209327: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instr

### Data Preprocessing

#### Resize images

In [2]:
# function to resize image into (dim1, dim2) given image df and path to images
def resize_images(image_df, read_path, write_path, dim1, dim2):
    tf.io.gfile.mkdir(write_path)
    
    for filename in image_df['filename']:
        # get the image path and raw data
        img_path = f"{read_path}/{filename}"
        img_data = tf.io.read_file(img_path)

        # decode the image and resize it
        image_decoded = tf.image.decode_jpeg(img_data, channels=3)
        resized_image = tf.image.resize(image_decoded, [dim1, dim2])
        resized_image = tf.cast(resized_image, tf.uint8)

        # encode the image
        encoded_image = tf.image.encode_jpeg(resized_image)

        # write the image into the file
        resized_write_path = f"{write_path}/{filename}"
        #print(f"Writing image {resized_write_path}")
        tf.io.write_file(resized_write_path, encoded_image)

#### Normalize images

In [3]:
# function to normalize images into ranges [0, 1]
def normalize_images(image_df, dim1, dim2, channels, read_path):
    image_arr = np.zeros(shape=(image_df.shape[0], dim1, dim2, channels))

    for i, filename in enumerate(image_df['filename']):
        # get image path and raw data
        img_path = f"{read_path}/{filename}"
        img_data = tf.io.read_file(img_path)

        # decode the image and normalize it
        img_decoded = tf.image.decode_jpeg(img_data, channels=channels)
        img_norm = tf.cast(img_decoded, tf.float32)/255.0

        # append it to the numpy array
        image_arr[i] = img_norm.numpy()

    return image_arr

In [4]:
training_df = pd.read_csv('Training_set.csv')
training_df.head(n=2)

,filename,label
0,Image_1.jpg,SOUTHERN DOGFACE
1,Image_2.jpg,ADONIS


In [5]:
resize_images(training_df, "train", "resized_train", 128, 128)

I0000 00:00:1739711553.247691   44797 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 4143 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4050 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.9


In [6]:
training_dataset = normalize_images(training_df, 128, 128, 3, "resized_train")

In [7]:
encoder = LabelEncoder()
training_df['label_enc'] =  encoder.fit_transform(training_df['label'])
training_df.head(n=1)

,filename,label,label_enc
0,Image_1.jpg,SOUTHERN DOGFACE,66


In [8]:
training_df['label'].unique().shape

(75,)

In [9]:
train_x, test_x, train_y, test_y = train_test_split(training_dataset,
                                                    training_df['label_enc'],
                                                   test_size=0.2,
                                                   random_state=42)

In [10]:
print(train_x.shape)
print(train_y.shape)
print(test_x.shape)
print(test_y.shape)

(5199, 128, 128, 3)
(5199,)
(1300, 128, 128, 3)
(1300,)


In [11]:
print(np.isnan(train_x).sum(), np.isinf(train_x).sum())  # Should be 0, 0
print(np.isnan(train_y).sum(), np.isinf(train_y).sum())  # Should be 0, 0
print(np.min(training_dataset), np.max(training_dataset))

0 0
0 0
0.0 1.0


### CNN Model

In [12]:
model = tf.keras.Sequential([
    tf.keras.layers.InputLayer(shape=(128, 128, 3)),
    
    tf.keras.layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.MaxPooling2D((2, 2)),
    tf.keras.layers.Dropout(0.25),

    tf.keras.layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.MaxPooling2D((2, 2)),
    tf.keras.layers.Dropout(0.25),
    
    tf.keras.layers.Conv2D(128, (3, 3), activation='relu', padding='same'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.MaxPooling2D((2, 2)),
    tf.keras.layers.Dropout(0.25),

    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(512, activation='relu'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dropout(0.5),
    tf.keras.layers.Dense(training_df['label'].nunique(), activation='softmax')
])

In [13]:
model.compile(optimizer='adam', 
              loss='sparse_categorical_crossentropy', 
              metrics=['accuracy'])

In [16]:
history = model.fit(train_x, train_y,
                    batch_size=128,
                    epochs=30,
                    validation_split=0.2,
                    verbose=1)

Epoch 1/30


2025-02-16 18:45:24.552389: I external/local_xla/xla/stream_executor/cuda/cuda_asm_compiler.cc:397] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_2677', 16 bytes spill stores, 16 bytes spill loads

2025-02-16 18:45:24.838711: I external/local_xla/xla/stream_executor/cuda/cuda_asm_compiler.cc:397] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_2677', 1196 bytes spill stores, 1200 bytes spill loads

2025-02-16 18:45:24.918607: I external/local_xla/xla/stream_executor/cuda/cuda_asm_compiler.cc:397] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_2677', 876 bytes spill stores, 880 bytes spill loads

2025-02-16 18:45:24.924828: I external/local_xla/xla/stream_executor/cuda/cuda_asm_compiler.cc:397] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_2677', 300 bytes spill stores, 300 bytes spill loads

2025-02-16 18:45:25.240139: I external/local_xla/xla

33/33 ━━━━━━━━━━━━━━━━━━━━ 19s 164ms/step - accuracy: 1.0000 - loss: 0.0058 - val_accuracy: 0.4279 - val_loss: 2.6401
Epoch 2/30
33/33 ━━━━━━━━━━━━━━━━━━━━ 3s 91ms/step - accuracy: 1.0000 - loss: 0.0059 - val_accuracy: 0.4856 - val_loss: 2.4680
Epoch 3/30
33/33 ━━━━━━━━━━━━━━━━━━━━ 3s 90ms/step - accuracy: 1.0000 - loss: 0.0062 - val_accuracy: 0.5279 - val_loss: 2.1544
Epoch 4/30
33/33 ━━━━━━━━━━━━━━━━━━━━ 3s 91ms/step - accuracy: 1.0000 - loss: 0.0058 - val_accuracy: 0.5519 - val_loss: 1.9617
Epoch 5/30
33/33 ━━━━━━━━━━━━━━━━━━━━ 3s 91ms/step - accuracy: 1.0000 - loss: 0.0054 - val_accuracy: 0.5558 - val_loss: 1.9087
Epoch 6/30
33/33 ━━━━━━━━━━━━━━━━━━━━ 3s 90ms/step - accuracy: 1.0000 - loss: 0.0046 - val_accuracy: 0.5635 - val_loss: 1.8894
Epoch 7/30
33/33 ━━━━━━━━━━━━━━━━━━━━ 3s 90ms/step - accuracy: 0.9995 - loss: 0.0063 - val_accuracy: 0.5558 - val_loss: 2.0192
Epoch 8/30
33/33 ━━━━━━━━━━━━━━━━━━━━ 3s 91ms/step - accuracy: 0.9998 - loss: 0.0064 - val_accuracy: 0.5615 - val_loss: 

In [17]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 128, 128, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 128, 128, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 64, 64, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64, 64, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 64, 64, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 64, 64, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 32, 32, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 32, 32, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 32, 32, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 32, 32, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 16, 16, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 16, 16, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 32768)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 512)            │    16,777,728 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 512)            │         2,048 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 75)             │        38,475 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 50,734,243 (193.54 MB)

 Trainable params: 16,910,923 (64.51 MB)

 Non-trainable params: 1,472 (5.75 KB)

 Optimizer params: 33,821,848 (129.02 MB)